# Flow cytometry with `ov.flow` — from FCS to a gated report

`ov.flow` covers the part of cytometry that has no analogue anywhere else in omicverse: the
display scalings, compensation, and above all the **sequential gating hierarchy**.

Reading the file is not here — that is `ov.io.read_fcs`, because it is I/O.

> **The data in this tutorial is simulated.** Everything runs end to end on a synthetic `.fcs`
> written by the notebook itself, so you can execute it without a file of your own. Substitute
> your own path at step 1 and the rest is unchanged.

## 0 — Setup

In [1]:
import numpy as np
import pandas as pd

import omicverse as ov

print('omicverse', ov.__version__)
print('ov.flow exports:', len(ov.flow.__all__))

omicverse 2.2.4
ov.flow exports: 30


Everything is registered, so an agent (or `ov.find_function`) can discover it:

In [2]:
ov.find_function('荧光补偿')   # -> ov.flow.compensate

/Users/fernandozeng/.omicos/env/.venv/lib/python3.11/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)


/Users/fernandozeng/Desktop/analysis/omicos-project/omicverse-src/omicverse/_registry.py:261: UserWarning: Function 'omicverse.alignment.dada2.merge_pairs' is missing a docstring; agent help output may be limited.
  warnings.warn(
/Users/fernandozeng/Desktop/analysis/omicos-project/omicverse-src/omicverse/_registry.py:261: UserWarning: Function 'omicverse.alignment.dada2.make_seqtab' is missing a docstring; agent help output may be limited.
  warnings.warn(
/Users/fernandozeng/Desktop/analysis/omicos-project/omicverse-src/omicverse/_registry.py:261: UserWarning: Function 'omicverse.alignment.dada2.remove_chimeras' is missing a docstring; agent help output may be limited.
  warnings.warn(


🔍 Found 1 matching function(s):


1. 📦 omicverse.flow._compensate.compensate
   📝 Apply fluorescence compensation to a cytometry AnnData using a spillover matrix — the file's own $SPILLOVER by default, or one supplied explicitly. Only the detectors the matrix names are touched, so scatter and time pass through untouched. Refuses to compensate twice, and records what it did in uns['flow'].
   🏷️  Aliases: compensate, 补偿, 荧光补偿, spillover_correction, compensation
   📁 Category: flow


<function omicverse.flow._compensate.compensate(adata: 'Any', spillover: 'Optional[Union[pd.DataFrame, np.ndarray]]' = None, *, matrix_type: 'str' = 'spillover', channels: 'Optional[Sequence[str]]' = None, layer_original: 'Optional[str]' = 'uncompensated', inplace: 'bool' = True) -> 'Any'>

## 1 — Reading an FCS file

`ov.io.read_fcs` keeps the two things a cytometry file exists to carry and that a naive reader
throws away:

* **`$PnN` vs `$PnS`** — the DETECTOR (`FITC-A`) versus the MARKER on it (`CD3`). The same
  antibody sits on a different detector between panels, so conflating them is the classic FCS
  mistake.
* **`$SPILLOVER`** — the acquisition spillover matrix. Without it compensation is impossible
  later, and uncompensated fluorescence is not wrong-*looking*, it is wrong.

First, write a synthetic file so this notebook is self-contained.

In [3]:
import flowio

rng = np.random.default_rng(0)
n = 30_000

# Three populations: CD3-CD19+ (B), CD3+CD4+ (T helper), CD3+CD8+ (T cytotoxic)
which = rng.choice([0, 1, 2], n, p=[0.3, 0.45, 0.25])
neg = lambda k: rng.normal(80, 120, k)
pos = lambda k: rng.normal(1.2e4, 4e3, k)

cd3  = np.where(which == 0, neg(n), pos(n))
cd19 = np.where(which == 0, pos(n), neg(n))
cd4  = np.where(which == 1, pos(n), neg(n))
cd8  = np.where(which == 2, pos(n), neg(n))
fsc  = np.abs(rng.normal(6e4, 1.2e4, n))
fsc_h = fsc * rng.normal(1.0, 0.02, n)          # doublets break FSC-A vs FSC-H
ssc  = np.abs(rng.normal(3.5e4, 9e3, n))

true = np.column_stack([cd3, cd19, cd4, cd8])
SPILL = np.array([[1.00, 0.12, 0.03, 0.01],
                  [0.09, 1.00, 0.06, 0.02],
                  [0.02, 0.07, 1.00, 0.11],
                  [0.01, 0.03, 0.08, 1.00]])
observed = true @ SPILL                          # what the instrument actually records

fluor = ["FITC-A", "PE-A", "APC-A", "BV421-A"]
markers = ["CD3", "CD19", "CD4", "CD8"]
X = np.column_stack([fsc, fsc_h, ssc, observed]).astype(np.float32)
names = ["FSC-A", "FSC-H", "SSC-A"] + fluor

meta = {f"$P{i+4}S": m for i, m in enumerate(markers)}
meta["$SPILLOVER"] = ",".join(["4"] + fluor + [f"{v:g}" for v in SPILL.flatten()])

with open("demo.fcs", "wb") as fh:
    flowio.create_fcs(fh, X.flatten().tolist(), channel_names=names,
                      opt_channel_names=[None]*3 + markers, metadata_dict=meta)
print("wrote demo.fcs")

wrote demo.fcs


In [4]:
adata = ov.io.read_fcs('demo.fcs')
adata

AnnData object with n_obs × n_vars = 30000 × 7
    obs: 'sample'
    var: 'n', 'channel', 'marker', 'PnB', 'PnE', 'PnG', 'PnR'
    uns: 'meta', 'fcs'

`var` keeps the detector and the marker apart — note the index uses the marker where the file named one:

In [5]:
adata.var[['channel', 'marker']]

       channel marker
FSC-A    FSC-A       
FSC-H    FSC-H       
SSC-A    SSC-A       
CD3     FITC-A    CD3
CD19      PE-A   CD19
CD4      APC-A    CD4
CD8    BV421-A    CD8

And the acquisition spillover matrix is **parsed**, not left as the raw comma string that other readers hand back:

In [6]:
adata.uns['fcs']['spillover'].round(3)

         FITC-A  PE-A  APC-A  BV421-A
FITC-A     1.00  0.12   0.03     0.01
PE-A       0.09  1.00   0.06     0.02
APC-A      0.02  0.07   1.00     0.11
BV421-A    0.01  0.03   0.08     1.00

## 2 — Compensation

Every fluorochrome emits into detectors other than its own. Observed values are `true @ S`, so
recovering the truth is `observed @ inv(S)`.

`compensate` touches **only the detectors the matrix names** — scatter and time are not
fluorescence — and refuses to run twice, because double compensation leaves the numbers looking
perfectly plausible.

In [7]:
# The notebook knows the ground truth, so compensation can be checked rather
# than eyeballed: observed = true @ S, so compensating must return `true`.
uncomp = np.asarray(adata[:, ['CD3', 'CD19', 'CD4', 'CD8']].X).copy()

ov.flow.compensate(adata)
comp = np.asarray(adata[:, ['CD3', 'CD19', 'CD4', 'CD8']].X)

print('recovers the true signal:', np.allclose(comp, true, rtol=1e-3, atol=2.0))
print()
print('CD19 (PE) in the CD3-negative population — this is spillover, not biology:')
cd3_neg = which == 0
print(f'  true         {true[~cd3_neg, 1].mean():8.0f}')
print(f'  as recorded  {uncomp[~cd3_neg, 1].mean():8.0f}   <- CD3-FITC bleeding into PE')
print(f'  compensated  {comp[~cd3_neg, 1].mean():8.0f}')
print()
# Compare like with like: the layer keeps all channels, so index it the same way.
fsc_before = np.asarray(adata.layers['uncompensated'])[:, adata.var.index.get_loc('FSC-A')]
fsc_after = np.asarray(adata.X)[:, adata.var.index.get_loc('FSC-A')]
print('scatter untouched:', np.allclose(fsc_before, fsc_after))

recovers the true signal: True

CD19 (PE) in the CD3-negative population — this is spillover, not biology:
  true               80
  as recorded      2192   <- CD3-FITC bleeding into PE
  compensated        80

scatter untouched: True


In [8]:
try:
    ov.flow.compensate(adata)
except ValueError as e:
    print('refused, as it should be:\n ', str(e)[:150])

refused, as it should be:
  this object is already compensated. Compensating twice is silently destructive — the numbers stay plausible — so it has to be explicit: re-read the fi


## 3 — Display transforms

A linear axis cannot show compensated fluorescence: after compensation a real population sits
partly **below zero**. A log axis cannot represent that at all. The field's answer is a family of
biexponential scalings — logarithmic in the bright decades, quasi-linear through zero.

These are derived from the GatingML 2.0 / Parks 2006 specifications and verified bit-for-bit
(4.4e-16, i.e. 2x float64 epsilon) against the reference implementation.

In [9]:
logicle = ov.flow.make_transform('logicle', t=262144, w=0.5, m=4.5, a=0.0)

probe = np.array([-1000., -100., 0., 100., 1e3, 1e4, 1e5, 262144.])
pd.DataFrame({
    'data':    probe,
    'logicle': logicle.apply(probe).round(4),
    'log':     ov.flow.make_transform('log', t=262144, m=4.5).apply(probe).round(4),
}).set_index('data')

           logicle     log
data                      
-1000.0    -0.2321     NaN
-100.0      0.0090     NaN
 0.0        0.1111     NaN
 100.0      0.2132  0.2403
 1000.0     0.4543  0.4625
 10000.0    0.6838  0.6848
 100000.0   0.9069  0.9070
 262144.0   1.0000  1.0000

`log` is `NaN` at and below zero — which is the honest answer, and exactly why it is the wrong
scale for compensated data. Note that logicle maps data `0` to a *positive* scale value: that
point (`x1`) is the centre of the linear region.

## 4 — Gating

The strategy is a **tree that belongs to the analysis, not to the sample** — which is what makes one strategy applicable to a whole batch.

In [10]:
gs = (ov.flow.GatingStrategy('PBMC panel')
      .add_gate(ov.flow.RectangleGate(
          name='Cells', dims=('FSC-A', 'SSC-A'),
          bounds=((3e4, 1.1e5), (1.5e4, 6e4))))
      .add_gate(ov.flow.PolygonGate(
          name='Singlets', dims=('FSC-A', 'FSC-H'),
          vertices=np.array([[3e4, 2.6e4], [1.1e5, 1.02e5],
                             [1.1e5, 1.16e5], [3e4, 3.4e4]])), parent='Cells')
      .add_gate(ov.flow.RectangleGate(
          name='CD3+', dims=('CD3',), bounds=((0.45, None),),
          transforms={'CD3': logicle}), parent='Singlets')
      .add_gate(ov.flow.QuadrantGate(
          name='CD4/CD8', dims=('CD4', 'CD8'), dividers=(0.45, 0.45),
          transforms={'CD4': logicle, 'CD8': logicle},
          quadrant_names=('DN', 'CD4+', 'CD8+', 'DP')), parent='CD3+'))

print(gs.tree())

root
  └ Cells
    └ Singlets
      └ CD3+
        └ CD4/CD8
        └ DN
        └ CD4+
        └ CD8+
        └ DP


In [11]:
res = gs.apply(adata)
res.stats()

  population    parent  count  parent_count  freq_parent  freq_total  low_n
0       CD3+  Singlets  20659         29361     0.703620    0.688633  False
1       CD4+      CD3+  13265         20659     0.642093    0.442167  False
2    CD4/CD8      CD3+  20659         20659     1.000000    0.688633  False
3       CD8+      CD3+   7327         20659     0.354664    0.244233  False
4      Cells      root  29363         30000     0.978767    0.978767  False
5         DN      CD3+     67         20659     0.003243    0.002233   True
6         DP      CD3+      0         20659     0.000000    0.000000   True
7   Singlets     Cells  29361         29363     0.999932    0.978700  False

`freq_parent` is the number people report — *"62% of CD3+"* — and it is meaningless without its
denominator, so the parent is named in its own column rather than left implicit. `low_n` flags
populations small enough that the percentage is noise: a frequency off 30 events has a 95% CI of
roughly ±18 points.

The masks are also written to `obs`, so gates compose with the rest of omicverse:

In [12]:
[c for c in adata.obs.columns if c.startswith('gate:')]

['gate:Cells',
 'gate:Singlets',
 'gate:CD3+',
 'gate:DN',
 'gate:CD4+',
 'gate:CD8+',
 'gate:DP',
 'gate:CD4/CD8']

## 5 — Interchange: Gating-ML 2.0

Gating-ML is the ISAC recommendation for handing a strategy to someone who does not use omicverse.

There is a trap in it. The spec types `gating:id` as `xs:ID` — an XML NCName — and almost no real
gate name is one:

In [13]:
pd.DataFrame({
    'gate name': ['CD3+', 'CD4+CD8-', 'Live cells', 'CD45RA+CCR7+', '4-1BB+', 'Singlets'],
}).assign(
    valid_xml_id=lambda d: d['gate name'].map(ov.flow.is_valid_xml_id),
    written_as=lambda d: d['gate name'].map(ov.flow.sanitize_id),
)

      gate name  valid_xml_id          written_as
0          CD3+         False             CD3_pos
1      CD4+CD8-         False      CD4_posCD8_neg
2    Live cells         False          Live_cells
3  CD45RA+CCR7+         False  CD45RA_posCCR7_pos
4        4-1BB+         False      gate_4-1BB_pos
5      Singlets          True            Singlets

`+` is the single most common character in a cytometry gate name. `ov.flow` therefore maps display
names onto generated valid ids and carries the human name in `custom_info`. `+`/`-` become
`pos`/`neg` rather than being stripped, because `CD4+CD8-` and `CD4-CD8+` are **opposite
populations** and silently merging them would be the worst thing an interchange format could do.

In [14]:
ov.flow.write_gatingml(gs, 'strategy.xml')
back = ov.flow.read_gatingml('strategy.xml')

print('gates:      ', sorted(back.gates))
print('hierarchy:  ', all(back.parent_of(n) == gs.parent_of(n) for n in gs.gates))

r2 = back.apply(adata.copy(), write_obs=False)
print('same events:', all(np.array_equal(res.masks[k], r2.masks[k]) for k in res.masks))

gates:       ['CD3+', 'CD4/CD8', 'Cells', 'Singlets']
hierarchy:   True


same events: True


## 6 — Unsupervised: FlowSOM

Automated clustering **complements** gating, it does not replace it: a gate is auditable and a
reviewer can argue with it; a metacluster is neither. Use it to find populations a strategy
missed, then draw the gate.

Cluster on transformed values and on the markers only — including scatter lets it dominate and you
get shape clusters rather than phenotypes.

In [15]:
adata.layers['logicle'] = np.column_stack([
    logicle.apply(np.asarray(adata[:, m].X).ravel()) for m in adata.var.index
]).astype(np.float32)

ov.flow.flowsom(adata, n_clusters=3, grid=(8, 8),
                markers=['CD3', 'CD19', 'CD4', 'CD8'],
                layer='logicle', random_state=0)

pd.crosstab(adata.obs['flowsom'], adata.obs['gate:CD3+'])

gate:CD3+  False   True
flowsom                
0            157   7491
1           8868     18
2            316  13150

The clusters recover the same split the gates found — from opposite directions, which is the point
of having both.

---

## What is not here

* **Reading FCS** — `ov.io.read_fcs`, because it is I/O.
* **Single-EV proteomics** — `ov.single.ev` owns `uns['ev']` and the vesicle vocabulary.
  `ov.single.ev.flowsom` and `ov.flow.flowsom` share one implementation.
* **Spectral unmixing** and **`.wsp` import** — deliberately out of scope for now.